# core

> Fill in a module description here

In [ ]:
# | default_exp api

In [ ]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [ ]:
# | export
from typing import Any, List, TypedDict, Tuple
from uuid import UUID
from product_horse.filesystems import (
    AbstractFileSystem,
    FilePathType,
)
from product_horse.db import (
    AbstractDatabase,
    UnvalidatedUser,
    UnvalidatedFileMetadata,
    User,
    FileMetadata,
    UnvalidatedTranscription,
    Transcription,
    Schema,
    UnvalidatedSchema,
)
from product_horse.audio_video import AudioEditor, AssemblyAiTranscript, Utterance
from product_horse.extraction import AIModelClient, QADataManager, Questions

In [ ]:
# | export

class FileDict(TypedDict):
    content: str
    name: str


def create_user_and_add_files(
    user_data: dict[str, Any],
    files: List[FileDict],
    db: AbstractDatabase,
    fs: AbstractFileSystem,
    authorized: bool = True,  # Is the user authorized to save these files?
) -> Tuple[User, List[FileMetadata]]:
    """
    Create a new user and add files for the user.

    Args:
        user_data (dict[str, Any]): User data dictionary.
        files (List[FileDict]): List of file dictionaries containing content and name.
        db (AbstractDatabase): Database instance for saving user and file metadata.
        fs (AbstractFileSystem): File system instance for saving files.
        authorized (bool, optional): Whether the files are authorized. Defaults to True.

    Returns:
        Tuple[User, List[FileMetadata]]: The created user object and a list of file metadata objects created.
    """

    unvalidated_user = UnvalidatedUser(**user_data)
    user: User = db.save_user(unvalidated_user)
    metadata: List[FileMetadata] = []

    for file_info in files:
        file_path = fs.build_user_path(user, FilePathType.USER_ID_BASE)
        fs.create_folder(file_path)
        file_content = file_info["content"].encode()
        file = fs.create_file(
            file_path, file_content, file_info["name"], authorized=authorized
        )

        # Save file metadata to database
        unvalidated_metadata = UnvalidatedFileMetadata(
            user_id=user.id, file_name=file_info["name"], file_path=file.uri
        )
        metadata.append(db.save_file_metadata(unvalidated_metadata))

    return user, metadata

In [ ]:
# | export

async def transcribe_file_and_extract_speakers(
    file_id: str,
    db: AbstractDatabase,
    audio_editor: AudioEditor,
) -> Tuple[Transcription, List[str]]:
    """
    Transcribe an audio file, extract speaker names, and save the transcription to the database.

    Args:
        file_id (str): The ID of the file to transcribe.
        db (AbstractDatabase): Database instance for saving transcription data.
        audio_editor (AssemblyAIClient): AssemblyAI client instance for transcription.

    Returns:
        Tuple[Transcription, List[str]]: The created transcription object and a list of extracted speaker names.
    """
    transcription_result: AssemblyAiTranscript = await audio_editor.transcribe(file_id)
    utterances: List[Utterance] = transcription_result.utterances
    speaker_names: List[str] = list(
        set([utterance.speaker for utterance in utterances])
    )

    # Future - update speaker names in the transcript using AI + find and replace

    # Save transcription to the database
    unvalidated_transcription = UnvalidatedTranscription(
        file_id=UUID(file_id), text=transcription_result.text
    )
    transcription: Transcription = db.save_transcription(unvalidated_transcription)

    return transcription, speaker_names

In [ ]:
# | export
def create_and_save_schema(
    text: str, db: AbstractDatabase, ai_model_client: AIModelClient
) -> Schema:
    """
    Create a schema from the given text using an AI model and save it to the database.

    Args:
        text (str): The text to create a schema from.
        db (AbstractDatabase): The database instance.
        ai_model_client (AIModelClient): The AI model client instance.

    Returns:
        Schema: The saved schema object.
    """
    questions = ai_model_client.create_schema(text)
    unvalidated_schema = UnvalidatedSchema(
        input_text=text, json_schema=questions.model_dump()
    )
    schema = db.save_schema(unvalidated_schema)
    return schema

In [ ]:
# | export
def extract_info_from_transcriptions(
    schema: Questions,
    transcriptions: List[Transcription],
    db: AbstractDatabase,
    ai_model_client: AIModelClient,
    output_file: str,
) -> bytes:
    """
    Extract information from transcriptions based on a schema and returns QADataManager

    Args:
        schema (Questions): The schema to use for extraction.
        transcriptions (List[Transcription]): The list of transcriptions to extract information from.
        db (AbstractDatabase): The database instance.
        ai_model_client (AIModelClient): The AI model client instance.
        output_file (str): The path to the output CSV file.
    
    Returns:
        bytes: The CSV in bytes.
    """
    qa_data_manager = QADataManager()

    for transcription in transcriptions:
        answers = ai_model_client.extract_information(transcription.text, schema)
        metadata = db.get_file_metadata(transcription.file_id)
        if metadata is None:
            raise ValueError(f"No metadata found for file_id {transcription.file_id}")
        qa_data_manager.add_answers(
            schema,
            answers,
            document_name=str(transcription.file_id),
            user_id=str(metadata.user_id),
        )
    return qa_data_manager.to_csv_bytes()



In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore